# htcie — From Handbook Lookup to Correlation Intelligence

## A narrative demo for thermal engineers

Every thermal engineer has been here: you need h, you reach for Dittus-Boelter, you write down a number. But is that the right number? Is it the best estimate? How much uncertainty are you carrying?

**htcie** was built to answer these questions transparently.

This demo follows an electronics cooling engineer who discovers that their single-correlation workflow was leaving insight — and accuracy — on the table. We will work through the same problem twice: first the traditional way, then with htcie. The difference is instructive.

---
**Problem:** Liquid cooling channels in a PCB cold plate. Water at 40°C, D = 6 mm, flow velocity 3 m/s. What is the convective heat transfer coefficient h?

In [1]:
import plotly.graph_objects as go

from htcie.core.loader import build_registry
from htcie.core.pipeline import run_evaluation
from htcie.core.state import (
    EngineeringState,
    FluidProperties,
    Geometry,
    BoundaryConditions,
    FlowState,
)

registry = build_registry()

## The Problem

Our engineer is designing a liquid-cooled cold plate for a high-power PCB assembly. The cooling channels are circular, D = 6 mm. The coolant is water at 40°C, with the following properties:

| Property | Symbol | Value | Units |
|---|---|---|---|
| Density | ρ | 992.2 | kg/m³ |
| Dynamic viscosity | μ | 6.51 × 10⁻⁴ | Pa·s |
| Thermal conductivity | k | 0.631 | W/m·K |
| Heat capacity | cp | 4179 | J/kg·K |

Flow velocity is 3.0 m/s. The channel wall is held at a controlled temperature (constant wall temperature boundary).

The engineer's current workflow: look up Dittus-Boelter, compute Re and Pr, plug in. Done.

Let's see what that gives — and what it misses.

In [2]:
# Water at 40 degrees C
rho, mu, k, cp = 992.2, 6.51e-4, 0.631, 4179.0
D, U = 0.006, 3.0

Re = rho * U * D / mu
Pr = cp * mu / k
f = (0.790 * __import__("math").log(Re) - 1.64) ** -2
n = 0.4  # heating
Nu_db = 0.023 * Re**0.8 * Pr**n
h_db = Nu_db * k / D

print(f"Re = {Re:.0f}, Pr = {Pr:.3f}")
print(f"Dittus-Boelter: Nu = {Nu_db:.1f}, h = {h_db:.1f} W/m\u00b2\u00b7K")
print("This is what most engineers stop at.")

Re = 27434, Pr = 4.311
Dittus-Boelter: Nu = 146.6, h = 15420.1 W/m²·K
This is what most engineers stop at.


## Differentiator 1 — Compare All Applicable Methods

Dittus-Boelter is one of many turbulent internal convection correlations. Depending on the flow regime, fluid, geometry, and boundary condition, other correlations may be equally or more valid — and may give meaningfully different results.

htcie checks every correlation in the registry automatically:

- Filters by geometry type (circular tube, flat plate, cylinder...)
- Filters by Reynolds number range
- Filters by Prandtl number range
- Filters by boundary condition type

Correlations that pass all checks are evaluated. Those that fail are excluded — with an explicit reason, never silently.

The result is a complete picture of what the literature says about this operating point — not just what one engineer remembered to look up.

In [3]:
state = EngineeringState(
    fluid=FluidProperties(
        density=992.2,
        viscosity=6.51e-4,
        thermal_conductivity=0.631,
        heat_capacity=4179.0,
    ),
    geometry=Geometry(
        geometry_type="circular_tube",
        characteristic_length=0.006,
        hydraulic_diameter=0.006,
    ),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=3.0),
)
report = run_evaluation(state, registry)
print(f"Applicable methods: {report.applicable}")
print("Excluded:")
for e in report.excluded:
    print(f"  {e['key']}: {e['reason']}")

Applicable methods: ['internal.dittus_boelter', 'internal.gnielinski', 'internal.petukhov', 'internal.sieder_tate']
Excluded:
  external.churchill_bernstein: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.hilpert: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.pohlhausen_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.turbulent_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.zukauskas_cylinder: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  internal.churchill_ozoe: Reynolds number 2.74e+04 is above maximum 2300.
  internal.shah_laminar: Reynolds number 2.74e+04 is above maximum 2300.
  tube_banks.grimison: Geometry type 'circular_tube' does not match required 'tube_bank'.
  tube_banks.zukauskas: Geometry type 'circular_tube' does not match required 'tube_bank'.


In [4]:
keys = [e["key"].split(".")[-1] for e in report.evaluations]
nus = [e["value"] for e in report.evaluations]
h_vals = [e["h"] for e in report.evaluations]

fig = go.Figure(
    [
        go.Bar(name="Nu", y=keys, x=nus, orientation="h", marker_color="#2980b9"),
    ]
)
fig.update_layout(
    title="Nusselt Number from All Applicable Correlations",
    xaxis_title="Nusselt number Nu",
    height=300,
)
fig.show()

## Differentiator 2 — Deterministic Ranking

With multiple applicable methods, which one should the engineer trust most? htcie scores each method on **8 deterministic factors**:

| Factor | What it captures |
|---|---|
| **Validity fit** | How close is the operating point to the center of the correlation's valid range? Near the boundary → penalised. |
| **Geometry match** | Does the correlation geometry exactly match the problem, or is it being applied by analogy? |
| **Regime match** | Does the correlation target the actual flow regime (laminar / transitional / turbulent)? |
| **Boundary condition match** | Constant wall temperature vs. constant heat flux — some correlations are specific to one. |
| **Correction completeness** | Does the correlation include viscosity correction, entrance length effects, or other important physics? |
| **Pedigree** | How extensively was the correlation validated in the original literature? |
| **Literature uncertainty** | Correlations with lower declared uncertainty score higher. |
| **Extrapolation penalty** | If any input is outside the stated valid range, a penalty is applied. |

Every factor is a deterministic function of the state and the correlation metadata. The ranking is fully reproducible — the same inputs always give the same ranking. This is essential for **audit trails**: if a design is reviewed six months later, the engineer can reproduce the ranking exactly and explain every factor.

No hidden weights. No magic numbers.

In [5]:
# Print ranking table
print(f"{'Rank':<5} {'Method':<35} {'Score':>7}  Breakdown")
print("-" * 80)
for i, r in enumerate(report.ranking, 1):
    breakdown_str = ", ".join(f"{k}={v:.2f}" for k, v in r["breakdown"].items())
    print(f"{i:<5} {r['key']:<35} {r['score']:>7.3f}  {breakdown_str}")

# Parallel coordinates plot — each line is a method, each axis is a score factor
if report.ranking:
    factors = list(report.ranking[0]["breakdown"].keys())
    scores = [r["score"] for r in report.ranking]
    method_names = [r["key"].split(".")[-1] for r in report.ranking]
    n = len(report.ranking)

    # First axis: correlation name (numeric index with named ticks)
    name_dim = dict(
        label="correlation",
        values=list(range(n)),
        range=[-0.5, n - 0.5],
        tickvals=list(range(n)),
        ticktext=method_names,
    )
    factor_dims = [
        dict(
            label=factor.replace("_", " "),
            values=[r["breakdown"].get(factor, 0) for r in report.ranking],
            range=[0.0, 1.0],
        )
        for factor in factors
    ]
    score_dim = dict(
        label="total score",
        values=scores,
        range=[min(scores) * 0.95, max(scores) * 1.05],
    )

    fig = go.Figure(
        go.Parcoords(
            line=dict(
                color=scores,
                colorscale="plasma",
                showscale=True,
                colorbar=dict(title="Score", thickness=12),
                cmin=min(scores) * 0.95,
                cmax=max(scores) * 1.05,
            ),
            dimensions=[name_dim] + factor_dims + [score_dim],
            labelfont=dict(size=10),
            tickfont=dict(size=9),
            rangefont=dict(size=8),
        )
    )
    fig.update_layout(
        title=dict(text="Ranking Score Breakdown by Factor", y=0.97, yanchor="top"),
        height=300,
        margin=dict(l=80, r=80, t=80, b=20),
    )
    fig.show()

Rank  Method                                Score  Breakdown
--------------------------------------------------------------------------------
1     internal.dittus_boelter               0.655  validity_fit=0.32, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=0.50, uncertainty_score=0.40, extrapolation_penalty=0.00
2     internal.gnielinski                   0.653  validity_fit=0.01, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=1.00, uncertainty_score=1.00, extrapolation_penalty=0.00
3     internal.petukhov                     0.652  validity_fit=0.01, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=1.00, uncertainty_score=1.00, extrapolation_penalty=0.00
4     internal.sieder_tate                  0.616  validity_fit=0.04, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=0.50, uncertainty_score=0.70, extra

## Differentiator 3 — Confidence and Spread

Knowing that h = 3200 W/m²·K is only useful if you also know **how sure you are**.

htcie quantifies uncertainty in two complementary ways:

**Literature uncertainty bands** — each correlation declares the uncertainty percentage from its original validation study. This propagates directly to h_low and h_high. A correlation validated to ±10% on smooth tubes is explicitly less certain than one validated to ±5% across a wider dataset.

**Inter-method spread** — when multiple methods agree closely (relative spread < 10%), confidence is high. When they disagree significantly, the engineer should investigate further. htcie reports this explicitly rather than hiding it.

These are independent signals. High confidence requires both low literature uncertainty *and* low inter-method spread.

In [6]:
keys = [e["key"].split(".")[-1] for e in report.evaluations]
h_vals = [e["h"] for e in report.evaluations]
h_low = [e["h_low"] or e["h"] for e in report.evaluations]
h_high = [e["h_high"] or e["h"] for e in report.evaluations]

error_minus = [h - lo if lo else 0 for h, lo in zip(h_vals, h_low)]
error_plus = [hi - h if hi else 0 for h, hi in zip(h_vals, h_high)]

fig = go.Figure(
    go.Bar(
        y=keys,
        x=h_vals,
        orientation="h",
        error_x=dict(
            type="data", symmetric=False, array=error_plus, arrayminus=error_minus
        ),
        marker_color="#3498db",
    )
)
fig.update_layout(
    title="Heat Transfer Coefficient h with Uncertainty Bands",
    xaxis_title="h [W/m\u00b2\u00b7K]",
    height=350,
)
fig.show()

print(f"Confidence class: {report.confidence}")
print(f"Inter-method spread: {report.spread['relative_spread']:.1%}")

Confidence class: high
Inter-method spread: 3.6%


## Differentiator 4 — Explainable Output

One of the most common sources of error in engineering calculations is the gap between what was computed and what was intended — a correlation applied slightly outside its valid range, an assumption that wasn't documented, a choice that can't be reconstructed.

htcie generates a **human-readable explanation** of every decision:

- Why the top method was selected
- What assumptions it makes
- Why each other method was excluded or ranked lower
- Any extrapolation warnings
- A plain-language recommendation

This explanation can be stored directly in the design file, embedded in a report, or used to justify method selection in a design review. The reasoning is transparent and auditable — not a black box.

In [7]:
print(report.explanation.to_text())

Recommended method: internal.dittus_boelter
Score: 0.655

Score breakdown:
  validity_fit: 0.317
  geometry_match: 1.000
  regime_match: 1.000
  boundary_match: 1.000
  correction_score: 0.500
  pedigree: 0.500
  uncertainty_score: 0.400
  extrapolation_penalty: 0.000

Confidence: high
Key assumptions:
  - Smooth circular tube, fully developed turbulent flow
  - Fluid properties evaluated at bulk mean temperature
  - $n = 0.4$ when fluid is being heated ($T_{wall} > T_{bulk}$); $n = 0.3$ when cooling
Excluded methods:
  external.churchill_bernstein: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.hilpert: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.pohlhausen_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.turbulent_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.zukauskas_cylinder: Geometry type 'circular_tube' does not match req

## The Insight

Starting from the same physical problem — D = 6 mm, water at 40°C, U = 3 m/s — the engineer now has:

- **All valid methods compared**, not just the one they remembered first
- **A deterministic, reproducible ranking** that can be regenerated from the same inputs at any future point
- **Quantified confidence and uncertainty bands** derived from literature, not guesswork
- **An audit trail** that explains every inclusion, exclusion, and score factor

One function call replaced what previously required cross-checking 3–4 handbooks, manually coding each correlation, estimating uncertainty by feel, and writing up a justification from scratch.

The output is not just a number. It is a structured, versioned, reproducible engineering decision.

In [8]:
import json

d = report.to_dict()
print(
    json.dumps(
        {
            "confidence": d["confidence"],
            "best_method": d["ranking"][0]["key"],
            "best_score": d["ranking"][0]["score"],
            "spread_pct": round(d["spread"]["relative_spread"] * 100, 1),
        },
        indent=2,
    )
)

{
  "confidence": "high",
  "best_method": "internal.dittus_boelter",
  "best_score": 0.655095,
  "spread_pct": 3.6
}
